In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import LabelEncoder


# Zadanie 1:  

## Email Spam

 Masz dane o 12 emailach z informacją czy to spam czy nie:

 **Zadania do wykonania:**

**a) Ręczne obliczenia**
1. Oblicz prawdopodobieństwa a priori: P(Spam=TAK) i P(Spam=NIE)
2. Dla każdej cechy oblicz prawdopodobieństwa warunkowe
3. Przewidź klasę dla nowego emaila:
```
   Słowo_1 = 'darmowy'
   Słowo_2 = 'wygrana'  
   Wykrzyknik = 'TAK'
```

Oblicz prawdopodobieństwa dla obu klas (TAK lub NIE) i znormalizuj

**b) Implementacja w Python**

1. Zaimplementuj obliczenia z punktu a) w Python (bez sklearn)
2. Porównaj wyniki z ręcznymi obliczeniami

**c) Sklearn**

1. Użyj `CategoricalNB` z sklearn do wytrenowania modelu
2. Porównaj wyniki z własnymi obliczeniami
3. Wyjaśnij różnice (jeśli są)

In [ ]:
data_spam = {
    'Słowo_1': ['darmowy', 'darmowy', 'spotkanie', 'raport', 'oferta', 'darmowy',
                'spotkanie', 'oferta', 'raport', 'darmowy', 'spotkanie', 'oferta'],
    'Słowo_2': ['wygrana', 'wygrana', 'jutro', 'kwartalny', 'specjalna', 'rabat',
                'dziś', 'limitowana', 'miesięczny', 'rabat', 'pilne', 'wyjątkowa'],
    'Wykrzyknik': ['TAK', 'TAK', 'NIE', 'NIE', 'TAK', 'TAK',
                   'NIE', 'TAK', 'NIE', 'TAK', 'NIE', 'TAK'],
    'Spam': ['TAK', 'TAK', 'NIE', 'NIE', 'NIE', 'TAK',
             'NIE', 'NIE', 'NIE', 'TAK', 'NIE', 'NIE']
}


In [ ]:
df = pd.DataFrame(data_spam)

priors = df["Spam"].value_counts(normalize=True)
print(priors)

features = ["Słowo_1", "Słowo_2", "Wykrzyknik"]
classes = df["Spam"].unique().tolist()

conditionals = {f: {c: {} for c in classes} for f in features}
for f in features:
    for c in classes:
        sub = df[df["Spam"] == c]
        denom = len(sub)
        for v in df[f].unique():
            conditionals[f][c][v] = (sub[f].eq(v).sum()) / denom

new_email = {"Słowo_1": "darmowy", "Słowo_2": "wygrana", "Wykrzyknik": "TAK"}
scores = {}
for c in classes:
    s = priors[c]
    for f in features:
        s *= conditionals[f][c].get(new_email[f], 0)
    scores[c] = s

total = sum(scores.values())
post = {c: (scores[c]/total if total else 0) for c in classes}
print(scores)
print(post)
print(max(post, key=post.get))

from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import CategoricalNB

X = df[features].copy()
le = {}
for col in features:
    enc = LabelEncoder()
    X[col] = enc.fit_transform(X[col])
    le[col] = enc

y_enc = LabelEncoder()
y = y_enc.fit_transform(df["Spam"])

model = CategoricalNB()
model.fit(X, y)

x_new = np.array([[le[col].transform([new_email[col]])[0] for col in features]])
pred = y_enc.inverse_transform(model.predict(x_new))[0]
proba = model.predict_proba(x_new)[0]
print(pred)
print({y_enc.inverse_transform([i])[0]: float(p) for i, p in enumerate(proba)})
